## Now we extend our project to implement Machine Learning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Import data
df_raw = pd.read_excel('financial_loan.xlsx')
df_raw.head(10)
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38576 entries, 0 to 38575
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     38576 non-null  int64         
 1   address_state          38576 non-null  object        
 2   application_type       38576 non-null  object        
 3   emp_length             38576 non-null  object        
 4   emp_title              37138 non-null  object        
 5   grade                  38576 non-null  object        
 6   home_ownership         38576 non-null  object        
 7   issue_date             38576 non-null  datetime64[ns]
 8   last_credit_pull_date  38576 non-null  datetime64[ns]
 9   last_payment_date      38576 non-null  datetime64[ns]
 10  loan_status            38576 non-null  object        
 11  next_payment_date      38576 non-null  datetime64[ns]
 12  member_id              38576 non-null  int64         
 13  p

### Our goal is to do a binary classification with loan_status as the target (Charged Off: 0, Fully Paid: 1), we'll remove "Current" at the moment

In [3]:
# We make a copy first
df = df_raw.copy()

In [4]:
# We remove the rows with loan status: current
df = df.loc[df['loan_status'] != "Current"]
df['loan_status']

0        Charged Off
1         Fully Paid
2        Charged Off
3         Fully Paid
4         Fully Paid
            ...     
38553     Fully Paid
38554     Fully Paid
38555     Fully Paid
38556     Fully Paid
38557     Fully Paid
Name: loan_status, Length: 37478, dtype: object

In [5]:
grade_map = {
    "G": 0,
    "F": 1,
    "E": 2,
    "D": 3,
    "C": 4,
    "B": 5,
    "A": 6
}
sub_grade_map = {
    "G5": 0,
    "G4": 1,
    "G3": 2,
    "G2": 3,
    "G1": 4,
    "F5": 5,
    "F4": 6,
    "F3": 7,
    "F2": 8,
    "F1": 9,
    "E5": 10,
    "E4": 11,
    "E3": 12,
    "E2": 13,
    "E1": 14,
    "D5": 15,
    "D4": 16,
    "D3": 17,
    "D2": 18,
    "D1": 19, "C5": 20, "C4": 21, "C3": 22, "C2": 23, "C1": 24,
    "B5": 25, "B4": 26, "B3": 27, "B2": 28, "B1":29, "A5": 30,
    "A4": 31, "A3": 32, "A2": 33, "A1": 34
}
df['grade'] = df['grade'].map(grade_map)
df['sub_grade'] = df['sub_grade'].map(sub_grade_map)

In [6]:
# We do an ordinal encoding for the emp_length, grade and sub-grade
emp_length_map = {
   "< 1 year": 0,
   "1 year": 1,
   "2 years": 2, 
   "3 years": 3,
   "4 years": 4,
   "5 years": 5,
   "6 years": 6,
   "7 years": 7,
   "8 years": 8,
   "9 years": 9,
   "10+ years": 10
}
df['emp_length'] = df['emp_length'].map(emp_length_map)
df['emp_length']

0         0
1         9
2         4
3         0
4        10
         ..
38553     4
38554     2
38555     4
38556    10
38557     2
Name: emp_length, Length: 37478, dtype: int64

### Now we remove the column which are known after the loan outcome and the ones which aren't related for our case

In [7]:
drop_cols = ['id', 'application_type', 'emp_title', 'last_credit_pull_date', 'last_payment_date', 'issue_date', 'next_payment_date', 'total_payment', 'member_id']
df = df.drop(columns=drop_cols, axis=1)
df.columns

Index(['address_state', 'emp_length', 'grade', 'home_ownership', 'loan_status',
       'purpose', 'sub_grade', 'term', 'verification_status', 'annual_income',
       'dti', 'installment', 'int_rate', 'loan_amount', 'total_acc'],
      dtype='object')

In [8]:
# Now we do the one-hot encoding for the categorical columns
df['verification_status']
cat_cols = ['address_state', 'home_ownership', 'verification_status', 'purpose']
df_encoded = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True
)
df_encoded.shape

(37478, 79)

In [9]:
# Since term column has only two values (36 and 60 months) and it is also a good criteria for our case, then we can do binary
# encoding together with our target (loan_status)
df_encoded['term'] = df_encoded['term'].astype(str).str.extract(r"(\d+)").astype(int)
df_encoded['term']

df_encoded['loan_status'] = df_encoded['loan_status'].map(
    {
        "Charged Off": 1,
        "Fully Paid": 0
    }
)
df_encoded['loan_status']

0        1
1        0
2        1
3        0
4        0
        ..
38553    0
38554    0
38555    0
38556    0
38557    0
Name: loan_status, Length: 37478, dtype: int64

In [10]:
df_encoded.shape

(37478, 79)

In [11]:
# Now we can start to split the target and feature columns. Target = df['loan_status']
X = df_encoded.drop(columns='loan_status', axis=1)
X.columns

y = df_encoded['loan_status']
X.shape, y.shape

((37478, 78), (37478,))

In [12]:
# Now we can see that the shape of our data is already correct, we can now split our data into train and test, normally it is 80% training and 20% test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=41, test_size=0.2, shuffle=True
)

In [13]:
# Before we can train the model, we need to scale the train data first, because logistic regression susceptible for scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
# We import logistic regression from linear model
from sklearn.linear_model import LogisticRegression
log_res_model = LogisticRegression(class_weight='balanced', random_state=0, max_iter=100)
log_res_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', random_state=0)

In [20]:
# Predict the score
y_pred = log_res_model.predict(X_test_scaled)
y_pred.shape

(7496,)

In [21]:
# We can check the score before we evaluate the model
log_res_model.score(X_test_scaled, y_test)

0.6642209178228389

In [22]:
# We can also evaluate deeper using sklearn metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[4310 2107]
 [ 410  669]]
              precision    recall  f1-score   support

           0       0.91      0.67      0.77      6417
           1       0.24      0.62      0.35      1079

    accuracy                           0.66      7496
   macro avg       0.58      0.65      0.56      7496
weighted avg       0.82      0.66      0.71      7496



In [23]:
# We use the roc_auc_score to observe the probability-based evaluation
y_prob = log_res_model.predict_proba(X_test_scaled)[:,1]
roc_auc_score(y_test, y_score=y_prob)

np.float64(0.6980783926153061)

### We can see that without considering class-imbalance (first model), we got a high accuracy but we missed the default, we only have 2% correct, which is risky. When we improve our model with weight='balanced we lost our accuracy but improved our recall, which reduce the default

In [66]:
# Can we improve our model? We can try Random Forest Classification
from sklearn.ensemble import RandomForestClassifier
rf_clf_model = RandomForestClassifier(n_estimators=100, min_samples_split=12, min_samples_leaf=12, class_weight='balanced', random_state=41, criterion='gini')

In [67]:
# Train the model
rf_clf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=12,
                       min_samples_split=12, random_state=41)

In [68]:
# Predict
y_pred_clf = rf_clf_model.predict(X_test_scaled)

In [69]:
# Check the score first before we go into further evaluation matrix
rf_clf_model.score(X_test_scaled, y_test)

0.7297225186766275

In [70]:
# Use the evaluation metrics
print("Classification Report\n----------------")
print(classification_report(y_test, y_pred=y_pred_clf))
print("Confusion Matrix\n------------------")
print(confusion_matrix(y_test, y_pred=y_pred_clf))

Classification Report
----------------
              precision    recall  f1-score   support

           0       0.90      0.77      0.83      6417
           1       0.27      0.51      0.35      1079

    accuracy                           0.73      7496
   macro avg       0.59      0.64      0.59      7496
weighted avg       0.81      0.73      0.76      7496

Confusion Matrix
------------------
[[4923 1494]
 [ 532  547]]


In [72]:
from sklearn.metrics import roc_auc_score

lr_auc = roc_auc_score(y_test, log_res_model.predict_proba(X_test_scaled)[:, 1])
rf_auc = roc_auc_score(y_test, rf_clf_model.predict_proba(X_test)[:, 1])
print(lr_auc, rf_auc)

0.6980783926153061 0.5354063428887268


/home/dio/Documents/python/ML-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


### We compared the logistic regression and random forest, more complex model does not guarantee a 'better' result or picture. Know the problem and the needs are important

In [78]:
# We can do one more model, which is Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    min_samples_split=4,
    min_samples_leaf=4,
    random_state=41
)

In [79]:
gb_model.fit(X_train_scaled, y_train)

GradientBoostingClassifier(min_samples_leaf=4, min_samples_split=4,
                           random_state=41)

In [80]:
y_pred = gb_model.predict(X_test_scaled)

In [81]:
gb_model.score(X_test_scaled, y_test)

0.8560565635005336

In [82]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      1.00      0.92      6417
           1       0.50      0.02      0.03      1079

    accuracy                           0.86      7496
   macro avg       0.68      0.51      0.48      7496
weighted avg       0.81      0.86      0.79      7496



In [83]:
print(confusion_matrix(y_test, y_pred))

[[6400   17]
 [1062   17]]
